## **Diagnóstico de cancro da mama com CNNs — Passive Learning (baseline)**
---
Inteligência Artificial Centrada no Humano

Faculdade de Ciências e Tecnologia da Universidade de Coimbra | Mestrado em Engenharia e Ciência de Dados

Treina a CNN com todo o conjunto de treino. Os resultados servem de referência para as experiências de Active Learning (`02_active_learning.ipynb`).

In [ ]:
# Configuração do ambiente: funciona no Google Colab e localmente
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/My Drive/miacd-iach-breast-cancer-diagnosis")  # pasta do repositório no Drive
    %pip install -q modAL-python scikeras
else:
    ROOT = Path.cwd().parent

sys.path.insert(0, str(ROOT))
DATA_PATH = ROOT / "data/breast-ultrasound-images-DatasetNinja/ds/img"
RESULTS_PATH = ROOT / "results"
RESULTS_PATH.mkdir(exist_ok=True)

In [ ]:
# Parâmetros
IMG_SIZE = 224          # reduzir (ex.: 128) se faltar memória
MAX_PER_CLASS = None    # limitar o nº de imagens originais por classe (None = todas)
EPOCHS = 15

### Dados

In [ ]:
# Carregar, dividir (treino/teste) e balancear o conjunto de treino
import numpy as np
from src.data import CLASSES, load_dataset

x_train, x_test, y_train, y_test = load_dataset(DATA_PATH, size=IMG_SIZE, max_per_class=MAX_PER_CLASS)

print(f"Conjunto de treino: {x_train.shape}, conjunto de teste: {x_test.shape}")
print(f"Memória das imagens: {(x_train.nbytes + x_test.nbytes) / 1e6:.0f} MB")
for split, y in [("Treino", y_train), ("Teste", y_test)]:
    print(split, {name: int(np.sum(y == i)) for i, name in enumerate(CLASSES)})

### Treino do modelo

In [ ]:
from tensorflow.keras.utils import to_categorical
from src.model import build_cnn

y_train_one_hot = to_categorical(y_train, num_classes=len(CLASSES))
y_test_one_hot = to_categorical(y_test, num_classes=len(CLASSES))

model = build_cnn(size=IMG_SIZE)
epochs_hist = model.fit(x_train, y_train_one_hot, epochs=EPOCHS, verbose=1, validation_split=0.2)

### Avaliação

In [ ]:
from src.evaluation import compute_metrics, plot_confusion_matrix, save_json

y_pred = np.argmax(model.predict(x_test), axis=1)
metrics = compute_metrics(y_test, y_pred)
metrics["accuracy_train"] = float(model.evaluate(x_train, y_train_one_hot, verbose=0)[1])
metrics["accuracy_test"] = metrics["accuracy"]

# Guardar para servir de baseline nos gráficos de Active Learning
save_json(metrics, RESULTS_PATH / "passive_learning.json")
metrics

In [ ]:
plot_confusion_matrix(y_test, y_pred, title="Passive Learning")